In [1]:
import numpy as np
import pandas as pd

import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import random_split

import tiktoken

In [2]:
enc = tiktoken.encoding_for_model("gpt-4o")

In [3]:
# yellow = 0
# green = 1
# blue = 2
# purple = 3

In [4]:
data = pd.read_csv('connections_history.csv')

In [5]:
data

,w1,w2,w3,w4,color,connection
0,fashion,manner,method,way,0,technique
1,crust,film,scum,skin,1,gross things that form on wet surfaces
2,catwalk,pit,stage,wings,2,parts of a theater
3,character,line,page,word,3,counted in document word counts
4,angel,babe,dove,lamb,0,symbols of innocence
...,...,...,...,...,...,...
4363,are,queue,sea,why,3,letter homophones
4364,hail,rain,sleet,snow,0,wet weather
4365,bucks,heat,jazz,nets,1,nba teams
4366,option,return,shift,tab,2,keyboard keys


In [6]:
data = data.dropna()

In [7]:
w1 = []
w2 = []
w3 = []
w4 = []

for row in data[['w1', 'w2', 'w3', 'w4']].values:
    value = enc.encode_batch(row)

    w1.append(value[0])
    w2.append(value[1])
    w3.append(value[2])
    w4.append(value[3])

In [8]:
enc_df = pd.DataFrame()

enc_df.insert(0, column='w1', value=w1)
enc_df.insert(1, column='w2', value=w2)
enc_df.insert(2, column='w3', value=w3)
enc_df.insert(3, column='w4', value=w4)
enc_df.insert(4, column='color', value=data.color)

In [9]:
enc_df

,w1,w2,w3,w4,color
0,[158179],"[76, 6237]",[8697],[3499],0.0
1,"[798, 570]",[40489],"[2786, 394]",[65435],1.0
2,"[8837, 26072]",[31300],[37244],"[86, 963]",2.0
3,[38245],[1137],[5342],[1801],3.0
4,[21887],"[65, 4905]","[67, 1048]","[75, 1855]",0.0
...,...,...,...,...,...
4350,[644],[9409],[25768],[60291],2.0
4351,[135735],[39775],"[82, 31953]",[149160],3.0
4352,[60984],[38002],"[73, 11083]",[88233],0.0
4353,[3941],[1034],[27472],[11957],1.0


In [10]:
train, val, test = random_split(enc_df, [0.7, 0.1, 0.2])

In [11]:
word_train, label_train = train.dataset[['w1', 'w2', 'w3', 'w4']], train.dataset.color
word_val, label_val = val.dataset[['w1', 'w2', 'w3', 'w4']], val.dataset.color
word_test, label_test = test.dataset[['w1', 'w2', 'w3', 'w4']], test.dataset.color

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(16, 1),
            nn.ReLU(inplace=True),
            nn.Linear(1, 1),
        )

    def forward(self, x):
        return self.classifier(x)

In [25]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.fc1 = nn.Sequential(nn.Linear(4, 4), nn.ReLU(inplace=True))
        self.fc2 = nn.Sequential(nn.Linear(4, 16), nn.ReLU(inplace=True))
        self.out = nn.Linear(16, num_classes)

    def forward(self, x):

        x = self.fc1(x)
        x = self.fc2(x)
        out = self.out(x)

        return out

In [26]:
cnn = SimpleCNN(num_classes=4)
print(cnn)

SimpleCNN(
  (fc1): Sequential(
    (0): Linear(in_features=4, out_features=4, bias=True)
    (1): ReLU(inplace=True)
  )
  (fc2): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU(inplace=True)
  )
  (out): Linear(in_features=16, out_features=4, bias=True)
)


In [27]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=0.01)

In [15]:
val = 16

word_tensor = torch.empty(size=(len(word_train), val))

for idx, row in enumerate(word_train.values):
    combo = row[0] + row[1] + row[2] + row[3]
    add_value = val - len(combo)
    combo = combo + np.repeat(-1, add_value).tolist()
    word_tensor[idx] = torch.Tensor(combo)

In [16]:
val = 1

label_tensor = torch.empty(size=(len(label_train), val))

for idx, row in enumerate(label_train.values):
    label_tensor[idx] = torch.Tensor([row])

In [ ]:
epochs = 5

for epoch in range(1, epochs + 1):
    cnn.train()

    for batch_idx, (data, labels) in enumerate(zip(word_tensor, label_tensor)):

        print(data.shape)
        print(labels.shape)

        optimizer.zero_grad()
        output = cnn(data)

        loss_value = loss_fn(output, labels)
        loss_value.backward()
        optimizer.step()

        if batch_idx % 100 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(word_train)}] Loss: {loss_value.item():.6f}')

    cnn.eval()
    # correct = 0
    # with torch.no_grad():
    #     for data, target in test_loader:
    #         data, target = data.to(device), target.to(device)
    #         output = cnn(data)
    #         pred = output.argmax(dim=1, keepdim=True)
    #         correct += pred.eq(target.view_as(pred)).sum().item()

    # print(f'\nAccuracy: {correct}/{len(test_loader.dataset)} ({100. * correct / len(test_loader.dataset):.1f}%)\n')

torch.Size([16])
torch.Size([1])


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x16 and 4x4)